In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
from tqdm import tqdm
import zipfile
import glob
import os

In [3]:
download_dir = os.path.join(os.getcwd(), "data")
os.makedirs(download_dir, exist_ok=True)

options = webdriver.ChromeOptions()
options.add_experimental_option("prefs", {
    "download.default_directory": download_dir,  # aquí se guardarán las descargas
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get("https://dados.gov.br/dados/conjuntos-dados/arboviroses-dengue")
time.sleep(5)  # espera a que cargue todo

# Encuentra todos los bloques de recursos
blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'row flex mb-5')]")

boton = driver.find_elements(By.ID, "btnCollapse")
boton = boton[2]
boton.click()
time.sleep(5)

for block in blocks:
    try:
        title = block.find_element(By.TAG_NAME, "h4").text.strip()
        formato = block.find_element(By.TAG_NAME, "span").text.strip()
        
        # Filtrar por año y formato CSV
        if any(str(year) in title for year in range(2020, 2027)) and formato == "CSV":
            print(f"Descargando: {title} ({formato})")
            button = block.find_element(By.XPATH, ".//button[contains(text(),'Acessar o recurso')]")
            button.click()
            time.sleep(2)  
    except Exception as e:
        print("Error en bloque:", e)

while True:
    noReady = [f for f in os.listdir(download_dir) if f.endswith(".crdownload")]
    #print(noReady)
    if len(noReady)==0:break
    time.sleep(10)

driver.quit()



Descargando: Dengue - 2026 (CSV)
Descargando: Dengue - 2025 (CSV)
Descargando: Dengue - 2024 (CSV)
Descargando: Dengue - 2023 (CSV)
Descargando: Dengue - 2022 (CSV)
Descargando: Dengue - 2021 (CSV)
Descargando: Dengue - 2020 (CSV)
['Sin confirmar 207806.crdownload', 'Sin confirmar 32282.crdownload', 'Sin confirmar 730082.crdownload', 'Sin confirmar 971132.crdownload', 'Sin confirmar 99785.crdownload']
['Sin confirmar 207806.crdownload', 'Sin confirmar 32282.crdownload', 'Sin confirmar 730082.crdownload', 'Sin confirmar 971132.crdownload', 'Sin confirmar 99785.crdownload']
['Sin confirmar 207806.crdownload', 'Sin confirmar 730082.crdownload', 'Sin confirmar 971132.crdownload', 'Sin confirmar 99785.crdownload']
['Sin confirmar 730082.crdownload', 'Sin confirmar 99785.crdownload']
['Sin confirmar 99785.crdownload']
['Sin confirmar 99785.crdownload']
[]


In [2]:
def unzip_all(input_folder="data", output_folder="data/csv"):
    # Crear carpeta de salida si no existe
    os.makedirs(output_folder, exist_ok=True)

    # Buscar todos los archivos .zip en la carpeta data/zip
    zips = [f for f in os.listdir(input_folder) if f.endswith(".zip")]

    print(f"Encontrados {len(zips)} archivos ZIP en {input_folder}")

    for z in tqdm(zips, desc="Descomprimiendo ZIPs", leave=False):
        zip_path = os.path.join(input_folder, z)
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(output_folder)
        except Exception as e:
            print(f"Error al descomprimir {z}: {e}")

    print(f"✅ Todos los ZIPs fueron extraídos en {output_folder}")



In [3]:
def csvMerge(input_folder="data/csv", output_folder="data", filename="dengue_brasil.csv"):
    # Crear carpeta de salida si no existe
    os.makedirs(output_folder, exist_ok=True)

    # Buscar todos los archivos CSV en la carpeta data/csv
    files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if "prepro" in f.lower() and f.endswith(".csv")]

    print(f"Encontrados {len(files)} archivos CSV en {input_folder}")

    val = True
    for f in tqdm(files, desc="Fusionando CSV", leave=False):
        try:
            df = pd.read_csv(f)
            if val:
                DF = df
                val = False
            else:
                DF = pd.concat([DF, df], ignore_index=True)
        except Exception as e:
            print(f"Error leyendo {f}: {e}")

    # Guardar el CSV final
    output_path = os.path.join(output_folder, filename)
    DF.to_csv(output_path, index=False)
    print(f"CSV fusionado guardado en {output_path}")


variable edad 

se suara la variable NU_IDAE_N si es prefijo 4 se sua la edad, si no, se usa 0

In [4]:
def transformar_edad(valor):
    try:
        valor_str = str(valor)
        if valor_str.startswith("4"):
            # Tomar los dos últimos dígitos
            return int(valor_str[-2:])
        else:
            return 0
    except:
        return 0

toma_muestra

In [5]:
def clasificar_muestra(valor):
    try:
        v = int(valor)
        if v in [1, 2, 3]:
            return 1
        elif v == 4:
            return 2
        else:
            return 0  # nan
    except:
        return None


hemorragicos

In [6]:
def marcar_obito(valor):
    # Si es NaN → 0, si no → 1
    return 0 if pd.isna(valor) else 1




### PIPE  LINE

In [7]:
unzip_all()

Encontrados 7 archivos ZIP en data


✅ Todos los ZIPs fueron extraídos en data/csv


In [8]:
# Lista de variables que quieres conservar
variables_sinan = [
    "CS_SEXO",
    "NU_IDADE_N",
    "SG_UF",
    "ID_MN_RESI",
    "SG_UF_NOT",
    "ID_MUNICIP",
    "ID_UNIDADE",
    "DT_SIN_PRI",
    "DIABETES",
    "HIPERTENSA",
    "ACIDO_PEPT",
    "RENAL",
    "AUTO_IMUNE",
    "HEPATOPAT",
    "CS_GESTANT",
    "DT_OBITO",
    "RESUL_PCR_",
    "CLASSI_FIN",
    "ALRM_SANG"
]+["SOROTIPO"]

# Ruta de los CSV originales
ruta = "data/csv"
archivos = glob.glob(os.path.join(ruta, "*.csv"))

for archivo in archivos:
    try:
        # Leer solo las columnas necesarias
        df = pd.read_csv(archivo, usecols=variables_sinan, low_memory=False)

        df = df.drop_duplicates()
        
        # Crear columnas derivadas
        df["TOMA_MUESTRA"] = df["RESUL_PCR_"].apply(clasificar_muestra)
        df["DEFUNCION"] = df["DT_OBITO"].apply(marcar_obito)
        df["EDAD"] = df["NU_IDADE_N"].apply(transformar_edad)

        #RESUL_PCR_
        df['SOROTIPO'].fillna(5,inplace=True)
        df["SOROTIPO"].replace({0.0:5.0},inplace=True)

        # Eliminar columnas originales
        df = df.drop(columns=["RESUL_PCR_", "DT_OBITO", "NU_IDADE_N"], errors="ignore")

        nombre_base = os.path.basename(archivo).replace(".csv", "_prepro.csv")
        salida = os.path.join(ruta, nombre_base)
        df.to_csv(salida, index=False)

        print(f"Procesado: {archivo} → {salida}")
    except ValueError as e:
        print(f"Error en {archivo}: {e}")



C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['SOROTIPO'].fillna(5,inplace=True)
C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, w

Procesado: data/csv\DENGBR20.csv → data/csv\DENGBR20_prepro.csv
Error en data/csv\DENGBR20_prepro.csv: Usecols do not match columns, columns expected but not found: ['DT_OBITO', 'NU_IDADE_N', 'RESUL_PCR_']


C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['SOROTIPO'].fillna(5,inplace=True)
C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, w

Procesado: data/csv\DENGBR21.csv → data/csv\DENGBR21_prepro.csv
Error en data/csv\DENGBR21_prepro.csv: Usecols do not match columns, columns expected but not found: ['DT_OBITO', 'NU_IDADE_N', 'RESUL_PCR_']


C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['SOROTIPO'].fillna(5,inplace=True)
C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, w

...

Procesado: data/csv\DENGBR24.csv → data/csv\DENGBR24_prepro.csv
Error en data/csv\DENGBR24_prepro.csv: Usecols do not match columns, columns expected but not found: ['DT_OBITO', 'NU_IDADE_N', 'RESUL_PCR_']


C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['SOROTIPO'].fillna(5,inplace=True)
C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, w

Procesado: data/csv\DENGBR25.csv → data/csv\DENGBR25_prepro.csv
Error en data/csv\DENGBR25_prepro.csv: Usecols do not match columns, columns expected but not found: ['DT_OBITO', 'NU_IDADE_N', 'RESUL_PCR_']


C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['SOROTIPO'].fillna(5,inplace=True)
C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\2793466283.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, w

Procesado: data/csv\DENGBR26.csv → data/csv\DENGBR26_prepro.csv
Error en data/csv\DENGBR26_prepro.csv: Usecols do not match columns, columns expected but not found: ['DT_OBITO', 'NU_IDADE_N', 'RESUL_PCR_']


In [9]:
csvMerge()

Encontrados 7 archivos CSV en data/csv


Fusionando CSV:  71%|███████▏  | 5/7 [00:20<00:10,  5.34s/it]C:\Users\EQUIPO\AppData\Local\Temp\ipykernel_17968\3036043148.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


CSV fusionado guardado en data\dengue_brasil.csv


In [10]:
df = pd.read_csv("data/csv/DENGBR26_prepro.csv")

In [11]:
df["TOMA_MUESTRA"].unique()

array([ 2.,  0.,  1., nan])

In [12]:
# Contar todos los valores en la columna, incluyendo NaN
print(df["TOMA_MUESTRA"].value_counts(dropna=False))


TOMA_MUESTRA
NaN    101238
2.0     54402
1.0      6727
0.0        33
Name: count, dtype: int64


In [49]:
df["TOMA_MUESTRA"].count()

np.int64(62181)

In [50]:
df.head()

,SG_UF_NOT,ID_MUNICIP,ID_UNIDADE,DT_SIN_PRI,CS_SEXO,CS_GESTANT,SG_UF,ID_MN_RESI,DIABETES,HEPATOPAT,RENAL,HIPERTENSA,ACIDO_PEPT,AUTO_IMUNE,RESUL_NS1,CLASSI_FIN,ALRM_SANG,TOMA_MUESTRA,DEFUNCION,EDAD
0,32,320500,2499495.0,2026-01-04,F,5.0,32,320500,2.0,2.0,2.0,1.0,2.0,2.0,2.0,10.0,0.0,2.0,0,41
1,32,320520,4676874.0,2026-01-05,F,5.0,32,320520,2.0,2.0,2.0,2.0,2.0,2.0,4.0,10.0,0.0,2.0,0,27
2,32,320500,2522772.0,2026-01-05,F,5.0,32,320500,2.0,2.0,2.0,2.0,2.0,2.0,2.0,10.0,0.0,NaN,0,34
3,32,320500,7838131.0,2026-01-04,M,6.0,32,320500,2.0,2.0,2.0,2.0,2.0,2.0,4.0,10.0,0.0,2.0,0,27
4,32,320530,11789.0,2026-01-05,M,6.0,32,320530,2.0,2.0,2.0,2.0,2.0,2.0,4.0,10.0,0.0,2.0,0,28


In [42]:
print(df.duplicated().sum())

2614


In [43]:
print(df[df.duplicated()].head())

     SG_UF_NOT  ID_MUNICIP  ID_UNIDADE  DT_SIN_PRI CS_SEXO  CS_GESTANT  SG_UF  \
33          32      320520   3405702.0  2026-01-05       M         6.0     32   
110         32      320500   2485958.0  2026-01-05       M         6.0     32   
306         32      320510   7329334.0  2026-01-13       M         6.0     32   
687         32      320530     11789.0  2026-01-23       F         5.0     32   
884         32      320500   2485958.0  2026-01-28       M         6.0     32   

     ID_MN_RESI  DIABETES  HEPATOPAT  RENAL  HIPERTENSA  ACIDO_PEPT  \
33       320520       2.0        2.0    2.0         2.0         2.0   
110      320500       2.0        2.0    2.0         2.0         2.0   
306      320130       2.0        2.0    2.0         2.0         2.0   
687      320530       2.0        2.0    2.0         2.0         2.0   
884      320500       2.0        2.0    2.0         2.0         2.0   

     AUTO_IMUNE  RESUL_NS1  CLASSI_FIN  ALRM_SANG  TOMA_MUESTRA  DEFUNCION  \
33      